In [18]:
# Importing the Libraries
import yfinance as yf
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import random
from collections import deque

In [6]:
# define stock symbol and time period
symbol = "AAPL"
start_date = "2020-01-01"
end_date = "2025-08-28"

In [7]:
# download historical data
data = yf.download(symbol, start=start_date, end=end_date)

C:\Users\Kamran\AppData\Local\Temp\ipykernel_22612\2071463506.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(symbol, start=start_date, end=end_date)
[*********************100%***********************]  1 of 1 completed


In [8]:
# Now we will alculate technical indicators that help AI agent make better trading decisions
data['SMA_5'] = data['Close'].rolling(window = 5).mean()
data['SMA_20'] = data['Close'].rolling(window = 20).mean()
data['Returns'] = data['Close'].pct_change()

In [9]:
# Now lets drop missing values and reset the index
data.dropna(inplace= True)
data.reset_index(drop= True, inplace= True)

In [10]:
# Now we will define the action space. The AI agent has three possible actions
ACTIONS = {0: 'Hold', 1: 'BUY', 2:'SELL'}

In [11]:
# Now, we will extract the State from the data
def get_state(data, index):
    return np.array([
        float(data.loc[index, 'Close']),
        float(data.loc[index, 'SMA_5']),
        float(data.loc[index, 'SMA_20']),
        float(data.loc[index, 'Returns'])
    ])

This function extracts the state representation from the dataset at a given time index. The state is an array containing:

- Closing price
- 5-day SMA
- 20-day SMA
- Daily return percentage 

This numerical representation of the stock market is fed into the AI model to make trading decisions.

Building The Trading Environment for our AI Agent

In [12]:
# We will now define a trading environment to interact with the Deep Q-Network (DQN) AI agent, which will allow it to learn how to trade stocks profitably
class TradingEnvironment:
    def __init__(self, data):
        self.data = data
        self.initial_balance = 10000
        self.balance = self.initial_balance
        self.holdings = 0
        self.index = 0

    def reset(self):
        self.balance = self.initial_balance
        self.holdings = 0
        self.index = 0
        return get_state(self.data, self.index)
    
    def step(self, action):
        price = float(self.data.loc[self.index, 'Close'])
        reward = 0

        if action == 1  and self.balance >= price: # BUY
            self.holdings = self.balance // price
            self.balance -= self.holdings * price
        elif action == 2 and self.holdings > 0: # SELL
            self.balance += self.holdings * price
            self.holdings = 0

        self.index += 1
        done = self.index >= len(self.data) -1

        if done:
            reward = self.balance - self.initial_balance

        next_state = get_state(self.data, self.index) if not done else None
        return next_state, reward, done, {}

The environment is implemented as a class that simulates the stock market. It tracks the agent’s balance, holdings, and current market index, and it provides new states and rewards in response to the agent’s actions

The Deep Q-Network (DQN)

In [13]:
class DQN(nn.Module):
    def __init__(self, state_size, action_size):
        super(DQN, self).__init__()
        self.fc1 = nn.Linear(state_size, 64)
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, action_size)

    def forward(self, x):
        x =torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

Here we built a Deep Q-Network using PyTorch to optimize stock trading decisions. The model features a three-layer neural network to predict trading actions, leveraging ReLU activation to enhance learning efficiency.

It outputs Q-values, which the agent utilizes to determine the best action: buy, sell, or hold, based on market conditions.

The DQN Agent

In [19]:
from scipy import optimize


class DQNAgent:
    def __init__(self, state_size, action_size):
        self.state_size = state_size
        self.action_size = action_size
        self.memory = deque(maxlen=2000)
        self.gamma = 0.95  # Discount factor
        self.epsilon = 1.0  # Exploration rate
        self.epsilon_min = 0.01
        self.epsilon_decay = 0.995
        self.learning_rate = 0.001
        self.model = DQN(state_size, action_size)
        self.optimizer = optim.Adam(self.model.parameters(), lr=self.learning_rate)
        self.criterion = nn.MSELoss()

    def remember(self, state, action, reward, next_state, done):
        self.memory.append((state, action, reward, next_state, done))

    def act(self, state):
        if random.uniform(0, 1) < self.epsilon:
            return random.choice(list(ACTIONS.keys()))
        state = torch.FloatTensor(state).unsqueeze(0)
        with torch.no_grad():
            q_values = self.model(state)
        return torch.argmax(q_values).item()

    def replay(self, batch_size):
        if len(self.memory) < batch_size:
            return
        minibatch = random.sample(self.memory, batch_size)

        for state, action, reward, next_state, done in minibatch:
            target = reward
            if not done:
                next_state_tensor = torch.FloatTensor(next_state).unsqueeze(0)
                target += self.gamma * torch.max(self.model(next_state_tensor)).item()

            state_tensor = torch.FloatTensor(state).unsqueeze(0)
            target_tensor = self.model(state_tensor).clone().detach()
            target_tensor[0][action] = target

            self.optimizer.zero_grad()
            output = self.model(state_tensor)
            loss = self.criterion(output, target_tensor)
            loss.backward()
            self.optimizer.step()

        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay

So, we developed a Deep Q-Learning Agent to interact with the stock market environment to enhance its decision-making through Experience Replay, which stores and reuses past experiences for training. The agent effectively balances Exploration vs. Exploitation, taking random actions initially and making smarter decisions as learning progresses.

Training is performed using batches of past experiences to refine the neural network’s performance. Additionally, a discount factor (gamma) is applied to weigh immediate and future rewards, to ensure long-term profitability.

Training the AI Agent

In [20]:
# train the agent
env = TradingEnvironment(data)
agent = DQNAgent(state_size=4, action_size=3)
batch_size = 32
episodes = 500
total_rewards = []

for episode in range(episodes):
    state = env.reset()
    done = False
    total_reward = 0

    while not done:
        action = agent.act(state)
        next_state, reward, done, _ = env.step(action)
        agent.remember(state, action, reward, next_state, done)
        state = next_state
        total_reward += reward

    agent.replay(batch_size)
    total_rewards.append(total_reward)
    print(f"Episode {episode+1}/{episodes}, Total Reward: {total_reward}")

print("Training Complete!")

C:\Users\Kamran\AppData\Local\Temp\ipykernel_22612\3280076810.py:4: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  float(data.loc[index, 'Close']),
C:\Users\Kamran\AppData\Local\Temp\ipykernel_22612\3280076810.py:5: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  float(data.loc[index, 'SMA_5']),
C:\Users\Kamran\AppData\Local\Temp\ipykernel_22612\3280076810.py:6: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  float(data.loc[index, 'SMA_20']),
C:\Users\Kamran\AppData\Local\Temp\ipykernel_22612\3280076810.py:7: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  float(data.loc[index, 'Returns'])
C:\Users\Kamra

Episode 1/500, Total Reward: -9843.833213806152
Episode 2/500, Total Reward: -9788.184329986572
Episode 3/500, Total Reward: -9851.251792907715
Episode 4/500, Total Reward: -9787.0658493042
Episode 5/500, Total Reward: -9923.657543182373
Episode 6/500, Total Reward: -9902.63236618042
Episode 7/500, Total Reward: -9839.966445922852
Episode 8/500, Total Reward: 40391.67359161377
Episode 9/500, Total Reward: -9893.37506866455
Episode 10/500, Total Reward: -9878.868843078613
Episode 11/500, Total Reward: -9782.63436126709
Episode 12/500, Total Reward: -9910.342971801758
Episode 13/500, Total Reward: -9875.208782196045
Episode 14/500, Total Reward: -9803.323768615723
Episode 15/500, Total Reward: -9876.719230651855
Episode 16/500, Total Reward: -9924.377952575684
Episode 17/500, Total Reward: -9936.82685470581
Episode 18/500, Total Reward: -9843.553085327148
Episode 19/500, Total Reward: -9860.412521362305
Episode 20/500, Total Reward: -9805.332298278809
Episode 21/500, Total Reward: -9802.

Here, we trained the AI Trading Agent using Deep Q-Learning, simulating 500 trading sessions where the agent learned from experience. It leveraged Exploration & Exploitation, initially taking random actions before making more informed decisions as training progressed.

Experience Replay is used to store past experiences, allowing the neural network to learn through batch training. Throughout the process, we tracked rewards to measure the agent’s performance improvements over time.

In [21]:
# create a fresh environment instance for testing
test_env = TradingEnvironment(data)
state = test_env.reset()
done = False

# simulate a trading session using the trained agent
while not done:
    # always choose the best action (exploitation)
    action = agent.act(state)
    next_state, reward, done, _ = test_env.step(action)
    state = next_state if next_state is not None else state

final_balance = test_env.balance
profit = final_balance - test_env.initial_balance
print(f"Final Balance after testing: ${final_balance:.2f}")
print(f"Total Profit: ${profit:.2f}")

C:\Users\Kamran\AppData\Local\Temp\ipykernel_22612\3280076810.py:4: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  float(data.loc[index, 'Close']),
C:\Users\Kamran\AppData\Local\Temp\ipykernel_22612\3280076810.py:5: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  float(data.loc[index, 'SMA_5']),
C:\Users\Kamran\AppData\Local\Temp\ipykernel_22612\3280076810.py:6: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  float(data.loc[index, 'SMA_20']),
C:\Users\Kamran\AppData\Local\Temp\ipykernel_22612\3280076810.py:7: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  float(data.loc[index, 'Returns'])
C:\Users\Kamra

Final Balance after testing: $138.02
Total Profit: $-9861.98


Summary    

So, in this article, we explored how to build an AI trading agent using Agentic AI and Deep Q-Learning, enabling it to make autonomous trading decisions. After training, our AI agent successfully generated a small but positive profit, which demonstrates its ability to navigate market fluctuations.